### Manual: 07-build-ops

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Compile the mmdet3d CUDA ops from the pinned tree (`python setup.py develop`).

All eleven extensions are required: `import mmdet3d.ops` pulls every one of
them, and the model needs `bev_pool` (camera branch) and `voxel_layer` (LiDAR
branch). The build is done with the CUDA 11.x toolkit chosen by the previous
cell; compatibility flags are passed because the pinned torch predates both
newer libstdc++ headers and newer host compilers.

Idempotent: skipped when `import mmdet3d.ops` already works.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

BUILD_WRAPPER = '''"""Compile the pinned mmdet3d CUDA extensions (setup.py develop)."""

import os
import runpy
import sys

repo = os.environ["REPO"]

if os.environ.get("SMOKE_BYPASS_CUDA_GUARD") == "1":
    import torch.utils.cpp_extension as cpp

    def _bypassed(self):
        print(
            "[smoke] WARNING: torch CUDA major-version guard bypassed "
            "(nvcc major != torch.version.cuda major). Documented deviation."
        )

    cpp.BuildExtension._check_cuda_version = _bypassed

os.chdir(repo)
sys.argv = ["setup.py", "develop"]
runpy.run_path(os.path.join(repo, "setup.py"), run_name="__main__")
'''

build_log = os.path.join(sc.LOGS, "build_ops.log")
report = {}

toolchain = (sc.state().get("build_toolchain") or {})
if not toolchain.get("cuda_home"):
    raise RuntimeError("no CUDA toolchain recorded; run the cuda toolchain cell first")
report["toolchain"] = toolchain

already = sc.capture('"%s" -c "import mmdet3d.ops"' % sc.PY)
report["import_before"] = "already importable" if "Error" not in already and "Traceback" not in already else already[-2000:]
report["skipped"] = report["import_before"] == "already importable"

arch_probe = (
    '"' + sc.PY + '" -c "import torch; a = torch.cuda.get_device_capability(0);'
    ' print(str(a[0]) + chr(46) + str(a[1]))"'
)
capacity = sc.capture(arch_probe).strip()
if not capacity or "." not in capacity or capacity.startswith("ERROR"):
    capacity = "7.5"
arch = capacity
report["torch_cuda_arch_list"] = arch
report["gpu"] = sc.capture(
    "nvidia-smi --query-gpu=index,name,compute_cap --format=csv,noheader"
)

if not report["skipped"]:
    sc.write_script("build_ops.py", BUILD_WRAPPER)
    env = sc.py_env(
        CUDA_HOME=toolchain["cuda_home"],
        PATH=toolchain["cuda_home"] + "/bin:" + os.environ.get("PATH", ""),
        FORCE_CUDA="1",
        MAX_JOBS=str(max(1, min(4, os.cpu_count() or 2))),
        TORCH_CUDA_ARCH_LIST=arch,
        # torch 1.10 predates newer libstdc++/gcc: add the headers that newer
        # <limits>/<cstdint> users assume, and let nvcc accept a newer host gcc.
        CXXFLAGS="-O2 -include limits -include cstdint",
        NVCC_PREPEND_FLAGS="--allow-unsupported-compiler -include limits -include cstdint",
    )
    if toolchain.get("cc"):
        env["CC"] = toolchain["cc"]
    if toolchain.get("cxx"):
        env["CXX"] = toolchain["cxx"]
    if toolchain.get("bypass_cuda_guard"):
        env["SMOKE_BYPASS_CUDA_GUARD"] = "1"
    report["env"] = dict(
        (key, env[key])
        for key in (
            "CUDA_HOME",
            "FORCE_CUDA",
            "MAX_JOBS",
            "TORCH_CUDA_ARCH_LIST",
            "CXXFLAGS",
            "NVCC_PREPEND_FLAGS",
            "CC",
            "CXX",
            "SMOKE_BYPASS_CUDA_GUARD",
        )
        if key in env
    )
    report["nvcc"] = sc.capture('"%s" --version' % os.path.join(toolchain["cuda_home"], "bin", "nvcc"))
    rc, text = sc.sh(
        '"%s" "%s"' % (sc.PY, os.path.join(sc.SCRIPTS, "build_ops.py")),
        log=build_log,
        env=env,
        cwd=sc.REPO,
        timeout=7200,
    )
    report["build_rc"] = rc
    if rc != 0:
        tail = sc.capture('tail -120 "%s"' % build_log)
        print("BUILD LOG TAIL:")
        print(tail)
        report["build_log_tail"] = tail[-6000:]

rc, text = sc.run_script("step_ops_import.py", log_name="step_ops_import.log", timeout=1800)
report["ops_check_rc"] = rc
report["ops_ok"] = "OPS_OK" in text
sc.save_json("build_ops.json", report)
sc.update(build_ops={"ok": report["ops_ok"], "arch": arch, "skipped": report["skipped"]})
print("BUILD_OPS_OK" if report["ops_ok"] else "BUILD_OPS_FAILED (log: %s)" % build_log)
